![DB Academy](./Includes/images/db-academy.png)

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">
<div style="background: #02A36F; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lab</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Build a Store Dimension</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Extend the dimensional model with a calculated column and run regional analytics.</div>
</div>
</div>


## REQUIRED: SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

This notebook runs on Serverless compute. Before running any cells, confirm Serverless is selected in the compute dropdown at the top-right of this notebook. This course was developed and tested on <strong>Serverless environment version 5</strong>.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this lesson. The setup script will initialize your catalog and schema so your queries run against the correct context.


In [0]:
%run ./Includes/Classroom-Setup-09

Workspace,User Catalog Name,Catalog Status
Learner Workspace,learn_get_started_dw_joemar_brillo,Catalog Already Exists


DataFrame[]

DataFrame[]

DataFrame[]

**Context.** In Lesson 08 you built the full star schema and ended by running an analytical query in the SQL Editor. In this lab you'll extend the model and answer richer questions using **Genie Code** to write most of the SQL, and the **SQL Editor** for one task.

The lab has **five tasks** that extend the star schema and analyze it from several angles.

**How to work each task:** the default path is **Genie Code** : open it inline with **Cmd/Ctrl + I** in the target cell (or the full **Genie / Assistant** panel), give it the prompt provided, run the result, and expand the reference to compare. You can always write the SQL yourself.

### Task 1: Confirm the starting state

Confirm that `dim_store` exists with 30 rows from Lesson 08.

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_109_1').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;"><code id="gp_109_1">Count the rows in the dim_store table.</code></pre>
</div>

In [0]:
%sql
-- Use Genie Code with the prompt above, then run it.
SELECT COUNT(*) FROM dim_store;

COUNT(*)
30


<details>
<summary><strong>Here is the count query</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb_t1').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb_t1">SELECT COUNT(*) AS row_count FROM dim_store;</code></pre>
</div>
</details>

You should see `30`. If you see `0` or a "table not found" error, the setup backfill rebuilds the model automatically, re-run the setup cell at the top and try again.

### Task 2: Regional revenue

Find **total revenue by region for 2024**, highest first. This joins `fact_orders` to `dim_store` (for `region`) and `dim_date` (to filter the year).

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_109_2').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;"><code id="gp_109_2" style="white-space:pre-wrap; overflow-wrap:break-word;">Using fact_orders joined to dim_store and dim_date on their surrogate keys, show each store region and its total revenue (sum of total_amount, rounded to 2 decimals) for the year 2024. Group by region and sort highest revenue first.</code></pre>
</div>

In [0]:
%sql
-- Use Genie Code with the prompt above, then run it.
SELECT
  ds.region,
  ROUND(SUM(fo.total_amount), 2) AS revenue
FROM fact_orders fo
JOIN dim_store ds ON fo.store_key = ds.store_key
JOIN dim_date  dd ON fo.date_key  = dd.date_key
WHERE dd.year = 2024
GROUP BY ds.region
ORDER BY revenue DESC;


region,revenue
West,1.7334421759E8
Midwest,1.2279596885E8
Southeast,1.2130746637E8
Northeast,1.0499109373E8


<details>
<summary><strong>Here is the regional revenue query</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb_t2').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb_t2">SELECT
  ds.region,
  ROUND(SUM(fo.total_amount), 2) AS revenue
FROM fact_orders fo
JOIN dim_store ds ON fo.store_key = ds.store_key
JOIN dim_date  dd ON fo.date_key  = dd.date_key
WHERE dd.year = 2024
GROUP BY ds.region
ORDER BY revenue DESC;</code></pre>
</div>
</details>

You should see four rows, one per region (`West`, `Midwest`, `Southeast`, `Northeast`). **West** tends to lead since it has the most stores (10), followed by Midwest and Southeast (7 each), then Northeast (6). Revenue scales roughly with store count.

### Task 3: Add a calculated `price_tier` column to `dim_product`

Classify each product by `list_price`:

- `Budget`: under $1,000
- `Mid-range`: $1,000 to $3,000 (inclusive)
- `Premium`: over $3,000

First add the column, then populate it with a `CASE` expression.

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_109_3').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;" ><code id="gp_109_3" style="white-space:pre-wrap; overflow-wrap:break-word;">On the dim_product table, add a STRING column called price_tier, then update it so list_price under 1000 is &#x27;Budget&#x27;, 1000 to 3000 inclusive is &#x27;Mid-range&#x27;, and over 3000 is &#x27;Premium&#x27;.</code></pre>
</div>

In [0]:
%sql
-- Add the price_tier column, then populate it with a CASE expression.
ALTER TABLE dim_product ADD COLUMNS (price_tier STRING);

UPDATE dim_product
SET price_tier = CASE
  WHEN list_price < 1000 THEN 'Budget'
  WHEN list_price <= 3000 THEN 'Mid-range'
  ELSE 'Premium'
END;

num_affected_rows
250


<details>
<summary><strong>Here are the full statements</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb_t3').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb_t3">ALTER TABLE dim_product ADD COLUMNS (price_tier STRING);
UPDATE dim_product
SET price_tier = CASE
  WHEN list_price &lt; 1000 THEN 'Budget'
  WHEN list_price &lt;= 3000 THEN 'Mid-range'
  ELSE 'Premium'
END;</code></pre>
</div>
</details>

Now verify the split. Run the count below.

In [0]:
%sql
SELECT price_tier, COUNT(*) AS product_count
FROM dim_product
GROUP BY price_tier
ORDER BY product_count DESC;

price_tier,product_count
Budget,125
Mid-range,69
Premium,56


You should see three rows. For the Vintage Audio catalog the split is **Budget = 125**, **Mid-range = 69**, **Premium = 56**. Budget is the largest tier (lots of lower-priced accessories), and the two higher tiers are smaller.

### Task 4: Run a query in the SQL Editor

Not every query belongs in a notebook. This task uses the **SQL Editor**, the same place you'll run ad-hoc analysis day to day.

**Do this:**

1. Copy the query below with the **Copy** button.
2. In the left sidebar, open **SQL Editor** (under *SQL*).
3. Make sure the editor's catalog is your `learn_get_started_dw_...` catalog and the schema is `vintage_audio` (set them in the schema browser or with a `USE` statement).
4. Paste the query, run it, and read the result right there in the editor.

The query answers: **revenue by price tier for 2024.**

<details>
<summary><strong>Here is the query to paste into the SQL Editor</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb_t4').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb_t4">SELECT
  dp.price_tier,
  COUNT(*)                       AS order_count,
  ROUND(SUM(fo.total_amount), 2) AS revenue
FROM fact_orders fo
JOIN dim_product dp ON fo.product_key = dp.product_key
JOIN dim_date    dd ON fo.date_key    = dd.date_key
WHERE dd.year = 2024
GROUP BY dp.price_tier
ORDER BY revenue DESC;</code></pre>
</div>
</details>

You should see three rows. `Budget` typically wins on **order count** (many inexpensive accessories), while the higher tiers contribute more **revenue per order**. The exact ordering reflects the Vintage Audio product mix.

### Task 5: Combine three dimensions

Now the kind of cross-tabulation a star schema makes easy: **2024 revenue by region and price tier**, sorted by region then revenue descending within each region. This touches the fact table plus three dimensions (store, product, date).

**Genie Code prompt:**

<div style="position:relative; margin:6px 0;">
<button onclick="var t=document.getElementById('gp_109_4').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:5px 11px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#EEF3F5; border-left:3px solid #1B5162; border-radius:6px; padding:12px 14px; margin:0; font-size:14px; white-space:pre-wrap;"><code id="gp_109_4" style="white-space:pre-wrap; overflow-wrap:break-word;">Using fact_orders joined to dim_store, dim_product, and dim_date, show store region, product price_tier, and total revenue (sum of total_amount rounded to 2 decimals) for 2024. Group by region and price_tier, and order by region then revenue descending.</code></pre>
</div>

In [0]:
%sql
-- Use Genie Code with the prompt above, then run it.
SELECT
  ds.region,
  dp.price_tier,
  ROUND(SUM(fo.total_amount), 2) AS revenue
FROM fact_orders fo
JOIN dim_store   ds ON fo.store_key   = ds.store_key
JOIN dim_product dp ON fo.product_key = dp.product_key
JOIN dim_date    dd ON fo.date_key    = dd.date_key
WHERE dd.year = 2024
GROUP BY ds.region, dp.price_tier
ORDER BY ds.region, revenue DESC;

region,price_tier,revenue
Midwest,Premium,6.840720229E7
Midwest,Mid-range,3.762621219E7
Midwest,Budget,1.676255437E7
Northeast,Premium,5.864004375E7
Northeast,Mid-range,3.217759085E7
Northeast,Budget,1.417345913E7
Southeast,Premium,6.71584392E7
Southeast,Mid-range,3.750065488E7
Southeast,Budget,1.664837229E7
West,Premium,9.725864854E7


<details>
<summary><strong>Here is the region-by-tier query</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cb_t5').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cb_t5">SELECT
  ds.region,
  dp.price_tier,
  ROUND(SUM(fo.total_amount), 2) AS revenue
FROM fact_orders fo
JOIN dim_store   ds ON fo.store_key   = ds.store_key
JOIN dim_product dp ON fo.product_key = dp.product_key
JOIN dim_date    dd ON fo.date_key    = dd.date_key
WHERE dd.year = 2024
GROUP BY ds.region, dp.price_tier
ORDER BY ds.region, revenue DESC;</code></pre>
</div>
</details>

You should see 12 rows: 4 regions times 3 price tiers. Same shape as the simpler queries - joins on surrogate keys, filter on a dimension attribute, aggregate a fact measure. Adding a third dimension only extended the `SELECT` and `GROUP BY`.

## Nice work
<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
<div style="color:#333;">
In this lab you:

- Confirmed the dimensional model from Lesson 08.
- Used <strong>Genie Code</strong> to write regional, tier, and multi-dimension revenue queries.
- Added a calculated <code>price_tier</code> column to <code>dim_product</code>.
- Ran a query in the <strong>SQL Editor</strong>, the day-to-day analysis surface.

</div>
</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>